# 05 · Production Drift Monitor
Detects distribution shift **before defects appear** — the early-warning system.

Three complementary detectors:
| Detector | Signal | Alert |
|---|---|---|
| PSI per feature | Tabular SET_/QUA_/ENV_ | PSI > 0.25 = RED |
| KS-test | Per-feature statistical | p < 0.05 = drift |
| Embedding drift | Cosine of mean fused emb | Cosine < 0.75 = YELLOW |

**Key feature**: Z01-Diff thermal ROI temperatures — encode injection dynamics,
degrade with tool wear *days before* visible defects.

In [ ]:
import sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent))
import config as C
from data.loaders  import load_parquet, build_cv_index, join_cv_to_parquet
from data.dataset  import get_tabular_cols, get_dxp_cols
from models.fusion import InjectionMouldingFusionModel
from training.splits import build_experiment_label_map, random_split
from analysis.drift import (
    compute_psi, psi_alert_level, ks_test_features,
    embedding_drift, DriftMonitor, ThermalDiffDriftDetector,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
df     = load_parquet(C.PARQUET_PATH)
cv_idx = build_cv_index()
df     = join_cv_to_parquet(df, cv_idx)
tabular_cols = get_tabular_cols(df)
train_df, _, test_df = random_split(df)
print(f"Reference: {len(train_df)} parts  |  Production: {len(test_df)} parts")

## 1 · PSI per Tabular Feature

In [ ]:
# Compute PSI for every tabular feature
ref_tab  = train_df[[c for c in tabular_cols if c in train_df.columns]].fillna(0).values
prod_tab = test_df[[c for c in tabular_cols if c in test_df.columns]].fillna(0).values

psi_vals = {}
for i, col in enumerate([c for c in tabular_cols if c in train_df.columns]):
    if i < ref_tab.shape[1] and i < prod_tab.shape[1]:
        psi_vals[col] = compute_psi(ref_tab[:, i], prod_tab[:, i])

psi_series = pd.Series(psi_vals).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(15, 5))
colours = [{"GREEN":"#66BB6A","YELLOW":"#FF9800","RED":"#EF5350"}[psi_alert_level(v)]
           for v in psi_series.values]
ax.bar(range(len(psi_series)), psi_series.values, color=colours, edgecolor="white", linewidth=0.5)
ax.axhline(0.1,  ls="--", color="#FF9800", linewidth=1.5, label="YELLOW (0.1)")
ax.axhline(0.25, ls="--", color="#EF5350", linewidth=1.5, label="RED (0.25)")
ax.set_xticks(range(len(psi_series)))
ax.set_xticklabels([c[:20] for c in psi_series.index], rotation=60, ha="right", fontsize=7)
ax.set_ylabel("PSI"); ax.set_title("Population Stability Index — All Tabular Features")
ax.legend()
plt.tight_layout(); plt.savefig(C.PLOTS_DIR/"drift_psi.png", dpi=150, bbox_inches="tight"); plt.show()

red_cols = psi_series[psi_series >= 0.25].index.tolist()
yel_cols = psi_series[(psi_series >= 0.1) & (psi_series < 0.25)].index.tolist()
print(f"RED  features ({len(red_cols)}):", red_cols[:5])
print(f"YELLOW features ({len(yel_cols)}):", yel_cols[:5])

## 2 · KS-Test Statistical Significance

In [ ]:
from scipy import stats

ks_results = {}
feat_cols  = [c for c in tabular_cols if c in train_df.columns and c in test_df.columns]
for i, col in enumerate(feat_cols):
    if i >= ref_tab.shape[1] or i >= prod_tab.shape[1]:
        continue
    stat, pval = stats.ks_2samp(ref_tab[:, i], prod_tab[:, i])
    ks_results[col] = {"stat": stat, "pval": pval, "drifted": pval < 0.05}

ks_df    = pd.DataFrame(ks_results).T
n_drift  = ks_df["drifted"].sum()
print(f"KS-test: {n_drift}/{len(ks_df)} features show statistically significant drift (p<0.05)")

# Plot top-10 drifted features
top_ks = ks_df.nlargest(10, "stat")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.barh(range(len(top_ks)), top_ks["stat"].values,
         color=["#EF5350" if d else "#42A5F5" for d in top_ks["drifted"]])
ax1.set_yticks(range(len(top_ks)))
ax1.set_yticklabels([c[:25] for c in top_ks.index])
ax1.set_xlabel("KS statistic"); ax1.set_title("Top-10 Drifted Features (KS statistic)")

ax2.scatter(ks_df["stat"], -np.log10(ks_df["pval"].clip(1e-10)),
            c=["#EF5350" if d else "#78909C" for d in ks_df["drifted"]], alpha=0.7, s=30)
ax2.axhline(-np.log10(0.05), ls="--", color="red", label="p=0.05")
ax2.set_xlabel("KS statistic"); ax2.set_ylabel("-log10(p-value)")
ax2.set_title("Volcano Plot: KS drift significance"); ax2.legend()

plt.tight_layout(); plt.show()

## 3 · Thermal ROI Drift — Sliding Window

In [ ]:
detector = ThermalDiffDriftDetector(train_df)
reports  = detector.batch_assess(test_df)

if reports:
    windows    = range(len(reports))
    max_psis   = [r["max_psi"] for r in reports]
    alert_cols = [{"GREEN":"#66BB6A","YELLOW":"#FF9800","RED":"#EF5350"}[r["alert"]]
                  for r in reports]

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(windows, max_psis, color=alert_cols, edgecolor="white", linewidth=0.5)
    ax.axhline(0.1,  ls="--", color="#FF9800", linewidth=1.5, label="YELLOW (0.1)")
    ax.axhline(0.25, ls="--", color="#EF5350", linewidth=1.5, label="RED (0.25)")
    ax.set_xlabel(f"Sliding window (size={detector.WINDOW})")
    ax.set_ylabel("Max PSI (thermal ROI)")
    ax.set_title("Thermal ROI Drift — Sliding Window over Production Sequence")
    ax.legend()
    plt.tight_layout()
    plt.savefig(C.PLOTS_DIR/"drift_thermal_window.png", dpi=150, bbox_inches="tight")
    plt.show()

    n_red = sum(1 for r in reports if r["alert"] == "RED")
    print(f"RED  windows: {n_red}/{len(reports)}")
    print(f"Top drifting features: {reports[-1].get('top_drift_features', [])}")
else:
    print("Insufficient thermal ROI data for sliding window (need >20 parts with ROI cols)")

## 4 · Embedding Drift

In [ ]:
from data.dataset import MultiModalDataset, collate_fn
from data.loaders import load_tabular_vector
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

scaler   = StandardScaler()
tab_vals = [load_tabular_vector(r, tabular_cols) for _, r in train_df.iterrows()]
scaler.fit(np.array(tab_vals))
tab_med  = {c: float(train_df[c].median()) for c in tabular_cols if c in train_df.columns}
roi_med  = {c: float(train_df[c].median()) for c in C.THERMAL_ROI_COLS if c in train_df.columns}

def _collect_embs(df_sub):
    ds = MultiModalDataset(df_sub, tabular_cols, get_dxp_cols(df_sub), train=False,
                           tabular_medians=tab_med, roi_medians=roi_med, tabular_scaler=scaler)
    ld = DataLoader(ds, batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=0)
    embs = []
    model.eval()
    with torch.no_grad():
        for batch in ld:
            b = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            embs.append(model(b)["fused_emb"].cpu().numpy())
    return np.concatenate(embs)

model = InjectionMouldingFusionModel(
    tabular_in_dim=len(tabular_cols), n_dxp_channels=len(get_dxp_cols(df)),
    n_experiments=len(build_experiment_label_map(df)),
).to(DEVICE)
ckpt = C.MODELS_DIR / "fusion_final.pt"
if ckpt.exists():
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

ref_embs  = _collect_embs(train_df)
prod_embs = _collect_embs(test_df)
result    = embedding_drift(ref_embs, prod_embs)

print("Embedding Drift Report:")
for k, v in result.items():
    print(f"  {k:25s}: {v}")

## 5 · Full DriftMonitor Report

In [ ]:
monitor = DriftMonitor(train_df, tabular_cols, ref_embeddings=ref_embs)
report  = monitor.check(test_df, prod_embeddings=prod_embs)

print(monitor.format_summary(report))
monitor.save_report(report, C.ARTIFACTS_DIR / "drift_report.json")
print("\nSaved:", C.ARTIFACTS_DIR / "drift_report.json")